Machine Learning Types 
1. Supervised Learning

Definition:
Model learns from labeled data (input + correct output).

Goal: Predict output for new unseen data.

Common Tasks:

Classification → Spam / Not Spam
Regression → House price prediction

Algorithms:

Linear Regression
Logistic Regression
Decision Trees
Random Forest

Example:
You train a model with past house prices → it predicts future prices.

2. Unsupervised Learning

Definition:
Model learns patterns from unlabeled data.

Goal: Discover hidden structure.

Common Tasks:

Clustering → Customer segmentation
Dimensionality Reduction → Feature reduction

Algorithms:

K-Means
Hierarchical Clustering
PCA

Example:
Grouping customers into similar behavior clusters.

3. Semi-Supervised Learning

Definition:
Uses small labeled + large unlabeled data.

Goal: Improve learning when labeling is expensive.

Example:
Few labeled medical images + many unlabeled → better diagnosis model.

4. Reinforcement Learning

Definition:
Model learns by trial and error using rewards.

Goal: Maximize cumulative reward.

Key Concepts:

Agent
Environment
Reward

Example:
Game playing, robotics, self-driving cars.

| Type            | Data Type    | Goal             | Example               |
| --------------- | ------------ | ---------------- | --------------------- |
| Supervised      | Labeled      | Predict output   | Price prediction      |
| Unsupervised    | Unlabeled    | Find patterns    | Customer segmentation |
| Semi-Supervised | Mixed        | Improve learning | Medical imaging       |
| Reinforcement   | Reward-based | Maximize reward  | Game AI               |

Train / Validation / Test Split (Correct Way):

    Standard Split
Train: 70%
Validation: 15%
Test: 15%

    Purpose
Train set: Learn patterns
Validation set: Tune model (hyperparameters)
Test set: Final unbiased evaluation

In [2]:
#1. IMPORTS
 
import pandas as pd
import numpy as np
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
 
#2. LOAD DATA
 
from sklearn.datasets import fetch_california_housing
 
data = fetch_california_housing(as_frame=True)
df = data.frame
 
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]
 
#3. CORRECT SPLIT (NO LEAKAGE)
 
#Step 1: Train vs Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
 
#Step 2: Validation vs Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
 
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
 
#4. PROPER SCALING (NO LEAKAGE)
 
scaler = StandardScaler()
 
#Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)
 
#Transform others
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
 
#5. MODEL TRAINING
 
model = LinearRegression()
model.fit(X_train_scaled, y_train)
 
#Validation performance
val_preds = model.predict(X_val_scaled)
val_mse = mean_squared_error(y_val, val_preds)
 
print("Validation MSE:", val_mse)
 
#Final test performance
test_preds = model.predict(X_test_scaled)
test_mse = mean_squared_error(y_test, test_preds)
 
print("Test MSE:", test_mse)

Train: (14448, 8)
Validation: (3096, 8)
Test: (3096, 8)
Validation MSE: 0.5408750691093342
Test MSE: 0.5202604958440161


In [3]:
#BAD PRACTICE
 
scaler = StandardScaler()
 
#Fitting on FULL data before split
X_scaled = scaler.fit_transform(X)
 
#Split after scaling → leakage
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
 
model_bad = LinearRegression()
model_bad.fit(X_train_bad, y_train_bad)
 
preds_bad = model_bad.predict(X_test_bad)
 
print("Leaky MSE:", mean_squared_error(y_test_bad, preds_bad))


Leaky MSE: 0.5558915986952442
